In [ ]:
# VectorDB에 pdf 파일 데이터를 저장 후 유사 문장 읽기
!pip install chromadb sentence-transformers pypdf

In [ ]:
import os, re
import uuid   # 고유 id 식별 생성
from typing import List
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
import pypdf

PDF_PATH = "sample.pdf"
CHROMA_DIR = ".pdf_demo"
COLLECTION = "pdf_coll"
MODEL_NAME = "all-MiniLM-L6-v2"

In [ ]:
# pdf파일에서 텍스트 추출
def extract_pdf_textFunc(path:str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"파일 없음:{path}")

    text_pages = []
    try:
        with open(path, 'rb') as f:
            reader = pypdf.PdfReader(f)
            for i, page in enumerate(reader.pages):
                txt = page.extract_text() or ""
                text_pages.append(txt)

        return "\n\n".join(text_pages)
    except Exception:
        raise RuntimeError("pdf 텍스트 추출 실패")


# 문단 분리 함수
def split_paragraphFunc(text:str, min_len=40) -> List[str]:
    paras = re.split(r"\n\s*\n+", text)   # 빈 줄 기준 분리
    paras = [re.sub(r"\s+", " ", p).strip() for p in paras]
    return [p for p in paras if len(p) >= min_len]

# 임베딩 함수
def embedderFunc(name:str= MODEL_NAME):
    return SentenceTransformer(name)

def embedFunc(model, texts:List[str]) -> List[List[float]]:
    return model.encode(texts, normalize_embeddings=True).tolist()

def get_collectionFunc(chroma_dir:str, name:str):
    client = PersistentClient(path=chroma_dir)
    return client.get_or_create_collection(name)

# 저장 함수
def upsert_pdfFunc(source_path:str):
    # 텍스트 추출
    full_text = extract_pdf_textFunc(source_path)   # pdf 파일 읽기
    if not full_text.strip():
        print("pdf에서 추출된 텍스트가 없어요")
        return 0

    # 문단 단위 청킹
    chunks = split_paragraphFunc(full_text, min_len=40)  # 문단 단위 분리
    if not chunks:
        print("저장할 문단 없음")
        return 0

    # 임베딩 처리
    model = embedderFunc(MODEL_NAME)
    embs = embedFunc(model, chunks)   # 각 청크를 벡터화

    ids = [str(uuid.uuid4()) for _ in chunks]  # 각 문단에 고유 id 부여
    # 기타 정보 (파일명, 문단 길이)
    metas = [{"source":os.path.basename(source_path), "len": len(c)} for c in chunks]

    col = get_collectionFunc(CHROMA_DIR, COLLECTION)
    col.add(
        ids=ids,
        documents=chunks,
        embeddings=embs,
        metadatas=metas
    )
    return len(chunks)


# 검색
def searchFunc(query:str, k:int):
    model = embedderFunc(MODEL_NAME)
    q_emb = embedFunc(model, [query])   # 질문 문장 임베딩
    # print(q_emb)

    collection = get_collectionFunc(CHROMA_DIR, COLLECTION)
    res = collection.query(query_embeddings=q_emb, n_results=k)
    # print(res)

    docs = res.get("documents", [[]])[0]   # [[]] 예외 방지용 패턴
    metas = res.get("metadatas", [[]])[0]
    ids = res.get("ids", [[]])[0]
    dists = res.get("distances", [[]])[0]

    for i, (doc, meta, _id, dist) in enumerate(zip(docs, metas, ids, dists), start=1):
        print(f"\n[{i}] id={_id}")
        print(f"source={meta.get('source')}, len={meta.get('len')}, distance={dist:.4f}")
        print(doc[:500] + ("..." if len(doc) > 500 else ""))


if __name__ == "__main__":
    n = upsert_pdfFunc(PDF_PATH)

    if n:
        print("\n검색 예")
        searchFunc("아내가 기침으로 쿨룩거리기는 벌써 달포가 넘었다.", k=3)